In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import warnings
from pytorch_dataset import EBSDStrainDataset

class ImprovedEBSDStrainCNN(nn.Module):
    """
    Upgraded CNN architecture with VGG-style deeper blocks
    and adaptive pooling to handle varying EBSD image resolutions.
    """
    def __init__(self):
        super(ImprovedEBSDStrainCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3)
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        return x.squeeze()

def calculate_metrics(y_true, y_pred, strain_classes=None):
    """
    Calculates percent error, standard deviation, and classification metrics
    by snapping continuous predictions to the nearest known discrete strain classes.
    """
    epsilon = 1e-8
    percent_errors = np.abs((y_true - y_pred) / (y_true + epsilon)) * 100
    avg_percent_error = np.mean(percent_errors)
    std_dev_error = np.std(percent_errors)

    if strain_classes is None:
        strain_classes = np.sort(np.unique(y_true))

    strain_to_int_map = {strain: i for i, strain in enumerate(strain_classes)}

    y_pred_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - pred))] for pred in y_pred])
    y_true_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - true))] for true in y_true])

    y_pred_classes_int = np.array([strain_to_int_map[s] for s in y_pred_snapped_floats])
    y_true_classes_int = np.array([strain_to_int_map[s] for s in y_true_snapped_floats])

    warnings.filterwarnings('ignore')
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_classes_int, y_pred_classes_int, average='weighted')

    return avg_percent_error, std_dev_error, precision, recall, f1

def train_and_evaluate(h5_path="ebsd_fcc_fe.h5", epochs=50, batch_size=32, learning_rate=0.0005):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"Loading data from: {h5_path}")

    train_dataset = EBSDStrainDataset(h5_path=h5_path, split="train")
    test_dataset = EBSDStrainDataset(h5_path=h5_path, split="test")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = ImprovedEBSDStrainCNN().to(device)

    criterion = nn.SmoothL1Loss()

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (patterns, strains, eulers) in enumerate(train_loader):
            patterns, strains = patterns.to(device), strains.to(device)

            noise = torch.randn_like(patterns) * 0.05
            patterns = patterns + noise

            optimizer.zero_grad()
            outputs = model(patterns)

            loss = criterion(outputs, strains)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        test_loss = 0.0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for patterns, strains, eulers in test_loader:
                patterns, strains = patterns.to(device), strains.to(device)
                outputs = model(patterns)

                loss = criterion(outputs, strains)
                test_loss += loss.item()

                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(strains.cpu().numpy())

        test_loss /= len(test_loader)
        scheduler.step()

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        avg_pe, std_pe, precision, recall, f1 = calculate_metrics(all_targets, all_preds)

        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {running_loss/len(train_loader):.4f} - Test Loss: {test_loss:.4f}")
        print(f"Metrics: P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f} | % Error: {avg_pe:.2f}% (Std: {std_pe:.2f})")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "best_ebsd_strain_model.pth")

    print("\nTraining completed. Best model saved to 'best_ebsd_strain_model.pth'.")

if __name__ == "__main__":
    train_and_evaluate(h5_path="/content/drive/MyDrive/ebsd_fcc_fe.h5", epochs=50, batch_size=64)


Using device: cuda
Loading data from: /content/drive/MyDrive/ebsd_fcc_fe.h5
Epoch [1/50] - Train Loss: 20.7997 - Test Loss: 11.5696
Metrics: P: 0.1465 | R: 0.1883 | F1: 0.1610 | % Error: 85.11% (Std: 103.20)
Epoch [2/50] - Train Loss: 10.2252 - Test Loss: 3.4212
Metrics: P: 0.5464 | R: 0.4908 | F1: 0.4728 | % Error: 45.56% (Std: 90.76)
Epoch [3/50] - Train Loss: 5.4352 - Test Loss: 3.1791
Metrics: P: 0.5154 | R: 0.4617 | F1: 0.4592 | % Error: 40.51% (Std: 66.80)
Epoch [4/50] - Train Loss: 4.6516 - Test Loss: 2.0364
Metrics: P: 0.6901 | R: 0.6458 | F1: 0.6519 | % Error: 26.55% (Std: 41.75)
Epoch [5/50] - Train Loss: 4.2049 - Test Loss: 1.7999
Metrics: P: 0.6946 | R: 0.6525 | F1: 0.6587 | % Error: 24.62% (Std: 37.34)
Epoch [6/50] - Train Loss: 3.9228 - Test Loss: 2.0141
Metrics: P: 0.6557 | R: 0.6308 | F1: 0.6317 | % Error: 31.42% (Std: 52.64)
Epoch [7/50] - Train Loss: 3.7100 - Test Loss: 1.6866
Metrics: P: 0.7075 | R: 0.6767 | F1: 0.6836 | % Error: 22.52% (Std: 29.65)
Epoch [8/50] - Tr